In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

STEP = 0.25

lon_min = -180.0
lat_min = -60.0

lon_bins = int(360 / STEP)
lat_bins = int(145 / STEP)

# 0 = empty
# 1 = no_land
# 2 = insufficient_veg
# 3 = no_forest_gain
# 4 = missing_s2
# 5 = valid

grid = np.zeros((lat_bins, lon_bins), dtype=np.uint8)

REJECTION_CODE = {
    "no_land": 1,
    "insufficient_veg": 2,
    "no_forest_gain": 3,
    "missing_s2": 4,
    "valid": 5,
}


def get_props(a):
    return a.get("properties", a)


def add_to_grid(aois):
    for a in aois:
        p = get_props(a)

        min_lon = p.get("minLon")
        max_lon = p.get("maxLon")
        min_lat = p.get("minLat")
        max_lat = p.get("maxLat")
        reason = p.get("rejection_reason", "")

        if None in (min_lon, max_lon, min_lat, max_lat):
            continue

        code = REJECTION_CODE.get(reason, 0)
        if code == 0:
            continue

        lon = (min_lon + max_lon) / 2
        lat = (min_lat + max_lat) / 2

        x = int((lon - lon_min) / STEP)
        y = int((lat - lat_min) / STEP)

        if 0 <= x < lon_bins and 0 <= y < lat_bins:
            grid[y, x] = code


with open("data/aois/valid_aois.json") as f:
    valid_aois = json.load(f)

with open("data/aois/rejected_aois.json") as f:
    rejected_aois = json.load(f)

add_to_grid(rejected_aois)
add_to_grid(valid_aois)

colors = [
    "#1a1a2e",  # 0 empty
    "#4a4a4a",  # 1 no_land
    "#e07b39",  # 2 insufficient_veg
    "#c0392b",  # 3 no_forest_gain
    "#8e44ad",  # 4 missing_s2
    "#27ae60",  # 5 valid
]

from matplotlib.colors import ListedColormap

cmap = ListedColormap(colors)

plt.figure(figsize=(20, 10), facecolor="#1a1a2e")
ax = plt.gca()
ax.set_facecolor("#1a1a2e")

ax.imshow(
    grid,
    origin="lower",
    interpolation="nearest",
    cmap=cmap,
    vmin=0,
    vmax=5,
)

legend_elements = [
    mpatches.Patch(color="#27ae60", label="Valid"),
    mpatches.Patch(color="#e07b39", label="Insufficient vegetation"),
    mpatches.Patch(color="#c0392b", label="No forest gain"),
    mpatches.Patch(color="#8e44ad", label="Missing S2"),
    mpatches.Patch(color="#4a4a4a", label="No land"),
]

ax.legend(
    handles=legend_elements,
    loc="lower left",
    framealpha=0.3,
    facecolor="#1a1a2e",
    edgecolor="white",
    labelcolor="white",
    fontsize=10,
)

ax.axis("off")
plt.tight_layout()
plt.savefig("data/aois/aoi_map.png", dpi=150, bbox_inches="tight", facecolor="#1a1a2e")
plt.show()

In [ ]:
import json
from pathlib import Path

BASE = Path("data/aois")

ALL_AOIS_FILE = BASE / "all_aois.json"
VALID_FILE = BASE / "valid_aois.json"
REJECTED_FILE = BASE / "rejected_aois.json"

CHECKPOINT_FILE = BASE / "aoi_filter_checkpoint.json"

with open(ALL_AOIS_FILE) as f:
    all_aois = json.load(f)

with open(VALID_FILE) as f:
    valid_aois = json.load(f)

with open(REJECTED_FILE) as f:
    rejected_aois = json.load(f)

all_ids = {
    a["properties"]["id"]
    for a in all_aois
}

valid_ids = {
    a["id"]
    for a in valid_aois
}

rejected_ids = {
    a["id"]
    for a in rejected_aois
}

done_ids = valid_ids | rejected_ids

missing_ids = all_ids - done_ids

checkpoint = {
    "valid": valid_aois,
    "rejected": rejected_aois,
}

with open(CHECKPOINT_FILE, "w") as f:
    json.dump(checkpoint, f)

print(f"All AOIs       : {len(all_ids):,}")
print(f"Valid AOIs     : {len(valid_ids):,}")
print(f"Rejected AOIs  : {len(rejected_ids):,}")
print(f"Done AOIs      : {len(done_ids):,}")
print(f"Missing AOIs   : {len(missing_ids):,}")

dupes = len(valid_ids) + len(rejected_ids) - len(done_ids)

print(f"Duplicate IDs  : {dupes:,}")

if dupes > 0:
    print("WARNING: duplicate AOI IDs detected")

print(f"\nCheckpoint written to:")
print(CHECKPOINT_FILE)

In [ ]:
import json
import statistics
from collections import Counter
from pathlib import Path


AOI_DIR = Path("data/aois")
VALID_PATH    = AOI_DIR / "valid_aois_enriched.json"
REJECTED_PATH = AOI_DIR / "rejected_aois_enriched.json"

def load(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path.resolve()}")
    with open(path) as f:
        return json.load(f)

def num_stats(values, label, unit=""):
    vals = [v for v in values if v is not None]
    if not vals:
        return {}
    vals_sorted = sorted(vals)
    n = len(vals)
    mean = sum(vals) / n
    med  = statistics.median(vals)
    sd   = statistics.stdev(vals) if n > 1 else 0
    return {
        "label": label,
        "unit":  unit,
        "n":     n,
        "min":   vals_sorted[0],
        "p25":   vals_sorted[int(n * 0.25)],
        "mean":  mean,
        "median":med,
        "p75":   vals_sorted[int(n * 0.75)],
        "max":   vals_sorted[-1],
        "stdev": sd,
    }

def fmt(v, decimals=4):
    if isinstance(v, float):
        return f"{v:>{12}.{decimals}f}"
    if isinstance(v, int):
        return f"{v:>12,}"
    return f"{str(v):>12}"

def print_stats(s):
    if not s:
        return
    unit_str = f" ({s['unit']})" if s['unit'] else ""
    print(f"\n  {s['label']}{unit_str}  [n={s['n']:,}]")
    print(f"    {'Min':<10} {fmt(s['min'])}")
    print(f"    {'P25':<10} {fmt(s['p25'])}")
    print(f"    {'Mean':<10} {fmt(s['mean'])}")
    print(f"    {'Median':<10} {fmt(s['median'])}")
    print(f"    {'P75':<10} {fmt(s['p75'])}")
    print(f"    {'Max':<10} {fmt(s['max'])}")
    print(f"    {'Std Dev':<10} {fmt(s['stdev'])}")

def section(title):
    width = 68
    print()
    print("=" * width)
    print(f"  {title}")
    print("=" * width)

def subsection(title):
    print(f"\n── {title} " + "─" * (60 - len(title)))

def pct(part, total):
    return 100 * part / total if total else 0

print("Loading data …", flush=True)
valid    = load(VALID_PATH)
rejected = load(REJECTED_PATH)
all_aois = valid + rejected

n_valid    = len(valid)
n_rejected = len(rejected)
n_total    = len(all_aois)

section("1. DATASET OVERVIEW")
print(f"\n  Total AOIs          : {n_total:>10,}")
print(f"  Valid               : {n_valid:>10,}  ({pct(n_valid,  n_total):.1f} %)")
print(f"  Rejected            : {n_rejected:>10,}  ({pct(n_rejected, n_total):.1f} %)")

section("2. REJECTION REASON BREAKDOWN")
reasons = Counter(r["rejection_reason"] for r in rejected)
print(f"\n  {'Reason':<25} {'Count':>10}  {'% of Rejected':>14}  {'% of Total':>10}")
print(f"  {'-'*25} {'-'*10}  {'-'*14}  {'-'*10}")
for reason, cnt in reasons.most_common():
    print(f"  {reason:<25} {cnt:>10,}  {pct(cnt, n_rejected):>13.1f}%  {pct(cnt, n_total):>9.1f}%")

section("3. GEOGRAPHIC COVERAGE")
for label, dataset in [("Valid AOIs", valid), ("Rejected AOIs", rejected), ("All AOIs", all_aois)]:
    lats = [r["centroid_lat"] for r in dataset]
    lons = [r["centroid_lon"] for r in dataset]
    subsection(label)
    print(f"    Latitude  range : {min(lats):>9.3f} → {max(lats):.3f}")
    print(f"    Longitude range : {min(lons):>9.3f} → {max(lons):.3f}")

section("4. REGIONAL DISTRIBUTION")
for label, dataset in [("Valid", valid), ("Rejected", rejected)]:
    subsection(label)
    regions = Counter(r.get("region", "Unknown") for r in dataset)
    n = len(dataset)
    print(f"  {'Region':<30} {'Count':>8}  {'%':>6}")
    print(f"  {'-'*30} {'-'*8}  {'-'*6}")
    for region, cnt in regions.most_common():
        print(f"  {region:<30} {cnt:>8,}  {pct(cnt,n):>5.1f}%")

section("5. BIOME DISTRIBUTION")
for label, dataset in [("Valid", valid), ("Rejected", rejected)]:
    subsection(label)
    biomes = Counter(r.get("biome_name", "Unknown") for r in dataset)
    n = len(dataset)
    print(f"  {'Biome':<45} {'Count':>8}  {'%':>6}")
    print(f"  {'-'*45} {'-'*8}  {'-'*6}")
    for biome, cnt in biomes.most_common():
        print(f"  {biome:<45} {cnt:>8,}  {pct(cnt,n):>5.1f}%")

section("6. NUMERIC STATISTICS — VALID AOIs")
numeric_fields = [
    ("aoi_area_km2",         "AOI Area",             "km²"),
    ("land_frac",            "Land Fraction",        "0–1"),
    ("veg_fraction",         "Vegetation Fraction",  "0–1"),
    ("forest_gain_area_km2", "Forest Gain Area",     "km²"),
    ("forest_gain_frac",     "Forest Gain Fraction", "0–1"),
    ("forest_gain_pixels",   "Forest Gain Pixels",   "px"),
]
for field, label, unit in numeric_fields:
    vals = [r[field] for r in valid if field in r]
    print_stats(num_stats(vals, label, unit))

section("7. NUMERIC STATISTICS — REJECTED AOIs")
for field, label, unit in numeric_fields:
    vals = [r[field] for r in rejected if field in r]
    print_stats(num_stats(vals, label, unit))

section("8. BOOLEAN FLAG SUMMARY")
flags = ["has_land", "has_s2", "has_veg", "has_gain"]
print(f"\n  {'Flag':<15} {'Valid (n / %)':<20} {'Rejected (n / %)':<20}")
print(f"  {'-'*15} {'-'*20} {'-'*20}")
for flag in flags:
    v_n = sum(1 for r in valid    if r.get(flag, 0) == 1)
    r_n = sum(1 for r in rejected if r.get(flag, 0) == 1)
    print(f"  {flag:<15} {f'{v_n:,} / {pct(v_n, n_valid):.1f}%':<20} {f'{r_n:,} / {pct(r_n, n_rejected):.1f}%':<20}")

section("9. FOREST GAIN — VALID vs REJECTED COMPARISON")
for label, dataset in [("Valid", valid), ("Rejected", rejected)]:
    areas = [r["forest_gain_area_km2"] for r in dataset if "forest_gain_area_km2" in r]
    total_gain = sum(areas)
    has_gain   = sum(1 for a in areas if a > 0)
    subsection(label)
    print(f"    Total forest gain area : {total_gain:>12,.2f} km²")
    print(f"    AOIs with any gain     : {has_gain:>12,}  ({pct(has_gain, len(areas)):.1f}%)")
    print(f"    Mean gain per AOI      : {total_gain/len(areas):>12.4f} km²")

print()
print("=" * 68)
print("  Done.")
print("=" * 68)
print()

In [ ]:
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

AOI_DIR       = Path("data/aois")
VALID_PATH    = AOI_DIR / "valid_aois_enriched.json"
REJECTED_PATH = AOI_DIR / "rejected_aois_enriched.json"
OUT_DIR       = Path("plots")
SCATTER_N     = 3000
RANDOM_SEED   = 42

OUT_DIR.mkdir(exist_ok=True)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

plt.rcParams.update({
    "figure.facecolor": "#0d1117",
    "axes.facecolor":   "#161b22",
    "axes.edgecolor":   "#30363d",
    "axes.labelcolor":  "#c9d1d9",
    "xtick.color":      "#8b949e",
    "ytick.color":      "#8b949e",
    "text.color":       "#e6edf3",
    "grid.color":       "#21262d",
    "grid.linestyle":   "--",
    "grid.alpha":       0.6,
    "legend.facecolor": "#1c2128",
    "legend.edgecolor": "#30363d",
    "font.family":      "monospace",
    "font.size":        10,
})

# Perceptually distinct, colourblind-safe (Wong 2011 + tweaks)
COLORS = {
    "valid":            "#2ecc71",   # vivid green
    "no_land":          "#3498db",   # blue
    "insufficient_veg": "#e67e22",   # orange
    "no_forest_gain":   "#e74c3c",   # red
    "missing_s2":       "#9b59b6",   # purple
}
REASON_LABELS = {
    "valid":            "Valid",
    "no_land":          "No Land",
    "insufficient_veg": "Insuff. Veg",
    "no_forest_gain":   "No Forest Gain",
    "missing_s2":       "Missing S2",
}
REASONS = ["valid", "no_land", "insufficient_veg", "no_forest_gain", "missing_s2"]

def threshold_vline(ax, x, label, color="#facc15", alpha=0.12, lw=1.8):
    ax.axvline(x, color=color, lw=lw, zorder=4)
    ax.axvspan(ax.get_xlim()[0], x, color=color, alpha=alpha, zorder=0)
    ymax = ax.get_ylim()[1]
    ax.text(x + (ax.get_xlim()[1] - ax.get_xlim()[0]) * 0.01, ymax * 0.96,
            label, color=color, fontsize=8, fontweight="bold", va="top", zorder=5)

def threshold_hline(ax, y, label, color="#facc15", alpha=0.12, lw=1.8):
    ax.axhline(y, color=color, lw=lw, zorder=4)
    ax.axhspan(ax.get_ylim()[0], y, color=color, alpha=alpha, zorder=0)
    xmax = ax.get_xlim()[1]
    ax.text(xmax * 0.01, y * 1.08, label, color=color,
            fontsize=8, fontweight="bold", va="bottom", zorder=5)

print("Loading data…")
with open(VALID_PATH) as f:
    valid = json.load(f)
with open(REJECTED_PATH) as f:
    rejected = json.load(f)

all_aois = valid + rejected
n_valid, n_rejected, n_total = len(valid), len(rejected), len(all_aois)
print(f"  {n_valid:,} valid  |  {n_rejected:,} rejected  |  {n_total:,} total")

v_sample = random.sample(valid,    min(SCATTER_N, n_valid))
r_sample = random.sample(rejected, min(SCATTER_N, n_rejected))

def savefig(name):
    p = OUT_DIR / name
    plt.savefig(p, dpi=150, bbox_inches="tight", facecolor=plt.gcf().get_facecolor())
    plt.close()
    print(f"  → {p}")

skip = {"Unknown", "N/A"}

def short_biome(b):
    return (b.replace("Tropical & Subtropical ", "T&S ")
             .replace("Temperate ", "Temp. ")
             .replace(", Savannas & Shrublands", "/Sav")
             .replace("Grasslands", "Grass")
             .replace("Forests/Taiga", "Forests")
             .replace(" & Mixed Forests", "")
             .replace("Woodlands & Scrub", "Scrub")
             .replace("Deserts & Xeric Shrublands", "Deserts/Xeric")
             .replace("Flooded Grasslands & Savannas", "Flooded Grass")
             .replace("Montane Grasslands & Shrublands", "Montane Grass")
             .replace(" Forests", ""))

print("\n[1/11] Density heatmaps…")

LAT_BINS, LON_BINS = 580, 1440  # 0.25° cells

def density_grid(records):
    grid = np.zeros((LAT_BINS, LON_BINS))
    for r in records:
        li = int(np.clip((r["centroid_lat"] + 60) / 0.25, 0, LAT_BINS - 1))
        lo = int(np.clip((r["centroid_lon"] + 180) / 0.25, 0, LON_BINS - 1))
        grid[li, lo] += 1
    return grid

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
fig.suptitle("AOI Spatial Density  (0.25° cells)", color="#e6edf3", fontsize=13, y=1.01)

specs = [
    (valid,    "Valid AOIs",    ["#0d2a1a", "#2ecc71"]),
    (rejected, "Rejected AOIs", ["#2a0d0d", "#e74c3c"]),
]
for ax, (records, title, clrs) in zip(axes, specs):
    grid = density_grid(records)
    cmap = LinearSegmentedColormap.from_list("", clrs)
    im = ax.imshow(
        np.flipud(grid ** 0.4),
        extent=[-180, 180, -60, 85],
        aspect="auto", cmap=cmap, interpolation="nearest",
    )
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="Count (γ=0.4)")
    ax.set_title(title, color="#e6edf3", fontsize=11)
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    ax.axhline(0, color="#30363d", lw=0.4)
    ax.axvline(0, color="#30363d", lw=0.4)
    ax.grid(True, lw=0.3)

plt.tight_layout()
savefig("01_density_heatmaps.png")

print("[2/11] Stacked bar by region…")

region_counts = defaultdict(Counter)
for r in all_aois:
    region_counts[r.get("region", "Unknown")][r.get("rejection_reason", "valid")] += 1

regions = [k for k in sorted(region_counts, key=lambda k: -sum(region_counts[k].values()))
           if k not in skip]

fig, ax = plt.subplots(figsize=(11, 5))
fig.suptitle("Rejection Composition by Biogeographic Realm", color="#e6edf3", fontsize=12)
bottoms = np.zeros(len(regions))
for reason in REASONS:
    vals = np.array([(region_counts[r].get(reason, 0) / sum(region_counts[r].values()))
                     for r in regions])
    ax.bar(regions, vals, bottom=bottoms, color=COLORS[reason],
           label=REASON_LABELS[reason], width=0.6)
    bottoms += vals

ax.set_ylabel("Proportion"); ax.set_ylim(0, 1)
ax.set_xticklabels(regions, rotation=25, ha="right")
ax.legend(loc="upper right", fontsize=9)
ax.yaxis.grid(True); ax.set_axisbelow(True)
plt.tight_layout()
savefig("02_stacked_bar_region.png")

print("[3/11] Stacked bar by biome…")

biome_counts = defaultdict(Counter)
for r in all_aois:
    biome_counts[r.get("biome_name", "Unknown")][r.get("rejection_reason", "valid")] += 1

biomes_all = [k for k in sorted(biome_counts, key=lambda k: -sum(biome_counts[k].values()))
              if k not in skip]

fig, ax = plt.subplots(figsize=(16, 5))
fig.suptitle("Rejection Composition by Biome", color="#e6edf3", fontsize=12)
bottoms = np.zeros(len(biomes_all))
for reason in REASONS:
    vals = np.array([(biome_counts[b].get(reason, 0) / sum(biome_counts[b].values()))
                     for b in biomes_all])
    ax.bar([short_biome(b) for b in biomes_all], vals, bottom=bottoms,
           color=COLORS[reason], label=REASON_LABELS[reason], width=0.6)
    bottoms += vals

ax.set_ylabel("Proportion"); ax.set_ylim(0, 1)
ax.set_xticklabels([short_biome(b) for b in biomes_all], rotation=35, ha="right", fontsize=8)
ax.legend(loc="upper right", fontsize=9)
ax.yaxis.grid(True); ax.set_axisbelow(True)
plt.tight_layout()
savefig("03_stacked_bar_biome.png")

print("[4/11] Filter funnel…")

funnel = {
    "All AOIs":        n_total,
    "Has Land":        n_total - sum(1 for r in rejected if r.get("rejection_reason") == "no_land"),
    "Has Vegetation":  n_total - sum(1 for r in rejected if r.get("rejection_reason") in {"no_land", "insufficient_veg"}),
    "Has Forest Gain": n_total - sum(1 for r in rejected if r.get("rejection_reason") in {"no_land", "insufficient_veg", "no_forest_gain"}),
    "Has S2 (Valid)":  n_valid,
}
labels = list(funnel.keys())
values = list(funnel.values())

fig, ax = plt.subplots(figsize=(8, 5))
fig.suptitle("AOI Filter Stage Funnel", color="#e6edf3", fontsize=12)

cmap_f = LinearSegmentedColormap.from_list("", ["#3498db", "#2ecc71"])
colors_f = [cmap_f(i / (len(labels) - 1)) for i in range(len(labels))]

max_val = max(values)

# Reverse for top-to-bottom funnel
labels_r = labels[::-1]
values_r = values[::-1]
colors_r = colors_f[::-1]

# Center each bar
lefts = [(max_val - v) / 2 for v in values_r]

bars = ax.barh(
    labels_r,
    values_r,
    left=lefts,
    color=colors_r,
    height=0.7
)

# Put count + percentage inside each bar
for bar, v in zip(bars, values_r):
    pct = 100 * v / n_total

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_y() + bar.get_height() / 2,
        f"{v:,}\n({pct:.1f}%)",
        ha="center",
        va="center",
        color="white",
        fontsize=9,
        fontweight="bold"
    )

ax.set_xlim(0, max_val)
ax.set_xlabel("AOI Count")

# Hide x-axis if you want a cleaner funnel look
ax.set_xticks([])

# Remove unnecessary spines
for spine in ["top", "right", "bottom"]:
    ax.spines[spine].set_visible(False)

ax.grid(False)

plt.tight_layout()
savefig("04_filter_funnel.png")

print("[5/11] Pairplot…")

PAIR_FIELDS = ["veg_fraction", "forest_gain_frac", "land_frac", "aoi_area_km2"]
PAIR_LABELS = ["Veg Fraction", "Gain Fraction", "Land Fraction", "AOI Area km²"]
N_PAIR = 800

vp = random.sample(v_sample, min(N_PAIR, len(v_sample)))
rp = random.sample(r_sample, min(N_PAIR, len(r_sample)))

def extract(records, fields):
    return np.array([[r.get(f, 0) for f in fields] for r in records])

Vmat = extract(vp, PAIR_FIELDS)
Rmat = extract(rp, PAIR_FIELDS)

n = len(PAIR_FIELDS)
fig, axes = plt.subplots(n, n, figsize=(12, 10))
fig.suptitle("Pairplot — Valid (green) vs Rejected (red)", color="#e6edf3", fontsize=12, y=1.01)

for i in range(n):
    for j in range(n):
        ax = axes[i][j]
        if i == j:
            ax.hist(Vmat[:, i], bins=30, color=COLORS["valid"], alpha=0.55, density=True)
            ax.hist(Rmat[:, i], bins=30, color=COLORS["no_forest_gain"], alpha=0.55, density=True)
        else:
            ax.scatter(Rmat[:, j], Rmat[:, i], c=COLORS["no_forest_gain"],
                       s=5, alpha=0.3, linewidths=0, rasterized=True)
            ax.scatter(Vmat[:, j], Vmat[:, i], c=COLORS["valid"],
                       s=5, alpha=0.4, linewidths=0, rasterized=True)

        if i == n - 1: ax.set_xlabel(PAIR_LABELS[j], fontsize=8)
        else:          ax.set_xticklabels([])
        if j == 0:     ax.set_ylabel(PAIR_LABELS[i], fontsize=8)
        else:          ax.set_yticklabels([])
        ax.tick_params(labelsize=7)

plt.tight_layout()
savefig("05_pairplot.png")

print("[6/11] Correlation heatmap…")

NUM_FIELDS  = ["aoi_area_km2", "land_frac", "veg_fraction",
               "forest_gain_frac", "forest_gain_area_km2"]
SHORT_NAMES = ["aoi_area", "land_frac", "veg_frac", "gain_frac", "gain_area"]

corr_data = np.array([[r.get(f, 0) for f in NUM_FIELDS] for r in valid], dtype=float)
C = np.corrcoef(corr_data.T)

# p-value approximation for significance stars (t-distribution)
from scipy import stats as _stats
n_obs = len(valid)
def pval(r, n):
    if abs(r) >= 1: return 0.0
    t = r * np.sqrt((n - 2) / (1 - r**2))
    return 2 * _stats.t.sf(abs(t), df=n - 2)

n_f = len(NUM_FIELDS)
CELL = 90   # px per cell → controls figure size
fig_px = CELL * n_f
fig, ax = plt.subplots(figsize=(fig_px / 72 + 2, fig_px / 72 + 1.5))
fig.suptitle("Correlation Matrix — Valid AOIs", color="#e6edf3", fontsize=13)

# Diverging blue→dark→green — clearly asymmetric around zero
cmap_corr = LinearSegmentedColormap.from_list("corr",
    ["#e74c3c", "#8e1c14", "#161b22", "#1a4d2a", "#2ecc71"])
im = ax.imshow(C, cmap=cmap_corr, vmin=-1, vmax=1, aspect="equal")
cbar = plt.colorbar(im, ax=ax, fraction=0.04, pad=0.03)
cbar.set_label("Pearson r", color="#c9d1d9")
cbar.ax.yaxis.set_tick_params(color="#8b949e")

for i in range(n_f):
    for j in range(n_f):
        r = C[i, j]
        p = pval(r, n_obs)
        star = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))
        # White text on dark cells, dark on bright cells
        txt_color = "#ffffff" if abs(r) > 0.35 else "#8b949e"
        ax.text(j, i, f"{r:.2f}", ha="center", va="center",
                fontsize=10, color=txt_color, fontweight="bold")
        if star and i != j:
            ax.text(j + 0.35, i - 0.32, star, ha="right", va="top",
                    fontsize=7, color=txt_color)

ax.set_xticks(range(n_f)); ax.set_yticks(range(n_f))
ax.set_xticklabels(SHORT_NAMES, rotation=35, ha="right", fontsize=10)
ax.set_yticklabels(SHORT_NAMES, fontsize=10)

# Gridlines between cells
for x in np.arange(-0.5, n_f, 1):
    ax.axhline(x, color="#0d1117", lw=1.5)
    ax.axvline(x, color="#0d1117", lw=1.5)

ax.text(0, -1.2, "* p<0.05  ** p<0.01  *** p<0.001",
        fontsize=8, color="#8b949e", transform=ax.transData)

plt.tight_layout()
savefig("06_corr_matrix.png")


print("[7/11] Biome x region heatmap…")

REGIONS_ORDERED = ["Palearctic", "Nearctic", "Neotropic", "Afrotropic",
                   "Indomalayan", "Australasia", "Oceania"]

biome_region = defaultdict(Counter)
for r in valid:
    b  = r.get("biome_name", "Unknown")
    rg = r.get("region", "Unknown")
    if b not in skip and rg in REGIONS_ORDERED:
        biome_region[b][rg] += 1

biomes_hr = sorted(biome_region, key=lambda b: -sum(biome_region[b].values()))
matrix = np.array([[biome_region[b].get(rg, 0) for rg in REGIONS_ORDERED]
                   for b in biomes_hr], dtype=float)

fig, ax = plt.subplots(figsize=(12, 7))
fig.suptitle("Valid AOI Count — Biome × Region", color="#e6edf3", fontsize=12)

cmap_hr = LinearSegmentedColormap.from_list("", ["#161b22", "#3498db"])
im = ax.imshow(matrix ** 0.5, cmap=cmap_hr, aspect="auto")
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="Count (γ=0.5)")

thresh = matrix.max() ** 0.5 * 0.5
for i, b in enumerate(biomes_hr):
    for j, rg in enumerate(REGIONS_ORDERED):
        v = int(matrix[i, j])
        if v > 0:
            tc = "#0d1117" if matrix[i, j] ** 0.5 > thresh else "#8b949e"
            ax.text(j, i, f"{v:,}", ha="center", va="center", fontsize=8, color=tc)

ax.set_xticks(range(len(REGIONS_ORDERED)))
ax.set_xticklabels(REGIONS_ORDERED, rotation=25, ha="right")
ax.set_yticks(range(len(biomes_hr)))
ax.set_yticklabels([short_biome(b) for b in biomes_hr], fontsize=9)
plt.tight_layout()
savefig("07_biome_region_heatmap.png")

print("[8/11] Cumulative forest gain by biome…")

biome_gain_total = Counter()
for r in valid:
    b = r.get("biome_name", "Unknown")
    if b not in skip:
        biome_gain_total[b] += r.get("forest_gain_area_km2", 0)

sorted_bg = sorted(biome_gain_total.items(), key=lambda x: -x[1])
bg_labels = [short_biome(b) for b, _ in sorted_bg]
bg_vals   = [v for _, v in sorted_bg]
total_g   = sum(bg_vals)
bg_cum    = np.cumsum(bg_vals) / total_g * 100

fig, ax1 = plt.subplots(figsize=(14, 5))
fig.suptitle("Cumulative Forest Gain by Biome — Valid AOIs", color="#e6edf3", fontsize=12)

cmap_bg = LinearSegmentedColormap.from_list("", ["#2ecc71", "#0d2a1a"])
bar_colors = [cmap_bg(i / max(len(bg_labels) - 1, 1)) for i in range(len(bg_labels))]
ax1.bar(range(len(bg_labels)), bg_vals, color=bar_colors, width=0.65)
ax1.set_ylabel("Total Gain Area (km²)")
ax1.set_xticks(range(len(bg_labels)))
ax1.set_xticklabels(bg_labels, rotation=35, ha="right", fontsize=8)
ax1.yaxis.grid(True); ax1.set_axisbelow(True)

ax2 = ax1.twinx()
ax2.plot(range(len(bg_labels)), bg_cum, color="#e67e22", lw=2,
         marker="o", ms=5, label="Cumulative %")
ax2.axhline(80, color="#9b59b6", lw=1.8, ls="--", label="80% threshold")
ax2.set_ylabel("Cumulative %", color="#e67e22")
ax2.tick_params(axis="y", colors="#e67e22")
ax2.set_ylim(0, 105)
ax2.legend(loc="center right", fontsize=9)
plt.tight_layout()
savefig("8_cumulative_gain_biome.png")


print("[9/11] Boxplot gain by biome…")

biome_gain_vals = defaultdict(list)
for r in valid:
    b = r.get("biome_name", "Unknown")
    if b not in skip:
        biome_gain_vals[b].append(r["forest_gain_frac"])

biomes_bp = sorted(biome_gain_vals, key=lambda b: -np.median(biome_gain_vals[b]))
bp_data   = [biome_gain_vals[b] for b in biomes_bp]

fig, ax = plt.subplots(figsize=(16, 6))
fig.suptitle("Forest Gain Fraction by Biome — Boxplot (valid AOIs)", color="#e6edf3", fontsize=12)

ax.boxplot(bp_data, positions=range(len(biomes_bp)),
           patch_artist=True, showfliers=False,
           medianprops=dict(color="#2ecc71", linewidth=2.5),
           boxprops=dict(facecolor="#1c2128", edgecolor="#3498db", linewidth=1.5),
           whiskerprops=dict(color="#8b949e", linewidth=1.2),
           capprops=dict(color="#8b949e", linewidth=1.2),
           widths=0.55)

means_bp = [np.mean(d) for d in bp_data]
ax.scatter(range(len(biomes_bp)), means_bp, color="#e67e22",
           zorder=5, s=40, label="Mean", marker="D")

ax.set_ylim(bottom=0)
ax.grid(True, axis="y"); ax.set_axisbelow(True)
threshold_hline(ax, 0.001, "min gain threshold", color="#facc15")

ax.set_xticks(range(len(biomes_bp)))
ax.set_xticklabels([short_biome(b) for b in biomes_bp], rotation=35, ha="right", fontsize=8)
ax.set_ylabel("Forest Gain Fraction")
ax.legend(fontsize=9)
plt.tight_layout()
savefig("9_boxplot_gain_biome.png")

print("\n[10/11] Biome contribution plots…")
print("\n[11/11] Region contribution plots…")

from scipy import stats as _stats

def contribution_figure(groupby_field, title_suffix, filename, label_fn=None):
    if label_fn is None:
        label_fn = lambda x: x

    valid_count = Counter()
    rejected_count = Counter()

    valid_gain = Counter()
    rejected_gain = Counter()

    for r in valid:
        g = r.get(groupby_field, "Unknown")
        if g in skip:
            continue

        valid_count[g] += 1
        valid_gain[g] += r.get("forest_gain_area_km2", 0)

    for r in rejected:
        g = r.get(groupby_field, "Unknown")
        if g in skip:
            continue

        rejected_count[g] += 1
        rejected_gain[g] += r.get("forest_gain_area_km2", 0)

    all_groups = sorted(
        set(valid_count) | set(rejected_count),
        key=lambda g: -(valid_count[g] + rejected_count[g])
    )

    short_labels = [label_fn(g) for g in all_groups]

    v_vals = np.array([valid_count[g] for g in all_groups])
    r_vals = np.array([rejected_count[g] for g in all_groups])

    vg_vals = np.array([valid_gain[g] for g in all_groups])
    rg_vals = np.array([rejected_gain[g] for g in all_groups])

    total_gain = vg_vals + rg_vals

    total = v_vals + r_vals
    rej_rate = np.where(total > 0, r_vals / total * 100, 0)

    x = np.arange(len(all_groups))

    fig, axes = plt.subplots(2, 2, figsize=(17, 10))
    fig.suptitle(
        f"AOI Contribution by {title_suffix}",
        color="#e6edf3",
        fontsize=13
    )

    ax = axes[0, 0]

    ax.bar(
        x,
        v_vals,
        color=COLORS["valid"],
        label="Valid",
        width=0.65,
    )

    ax.bar(
        x,
        r_vals,
        bottom=v_vals,
        color=COLORS["no_forest_gain"],
        label="Rejected",
        width=0.65,
        alpha=0.8,
    )

    ax.set_title("AOI Count — Valid vs Rejected", color="#e6edf3")
    ax.set_ylabel("Count")
    ax.set_xticks(x)
    ax.set_xticklabels(short_labels, rotation=35, ha="right", fontsize=8)
    ax.legend(fontsize=9)
    ax.yaxis.grid(True)
    ax.set_axisbelow(True)

    ax = axes[0, 1]

    bars = ax.bar(
        x,
        rej_rate,
        color=[plt.cm.RdYlGn_r(v / 100) for v in rej_rate],
        width=0.65,
    )

    ax.set_title("Rejection Rate (%)", color="#e6edf3")
    ax.set_ylabel("% Rejected")
    ax.set_ylim(0, 105)

    ax.set_xticks(x)
    ax.set_xticklabels(short_labels, rotation=35, ha="right", fontsize=8)

    for bar, val in zip(bars, rej_rate):
        if val > 5:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                val + 1,
                f"{val:.0f}%",
                ha="center",
                fontsize=7,
                color="#e6edf3",
            )

    ax.yaxis.grid(True)
    ax.set_axisbelow(True)

    ax = axes[1, 0]

    ax.bar(
        x,
        vg_vals,
        color="#2ecc71",
        label="Gain in Valid AOIs",
        width=0.65,
    )

    ax.bar(
        x,
        rg_vals,
        bottom=vg_vals,
        color="#e74c3c",
        alpha=0.85,
        label="Gain in Rejected AOIs",
        width=0.65,
    )

    ax.set_title(
        "Total Forest Gain Area (Valid + Rejected)",
        color="#e6edf3",
    )

    ax.set_ylabel("Gain Area (km²)")
    ax.set_xticks(x)
    ax.set_xticklabels(short_labels, rotation=35, ha="right", fontsize=8)
    ax.legend(fontsize=8)

    ax.yaxis.grid(True)
    ax.set_axisbelow(True)

    total_valid_gain = vg_vals.sum()
    total_rejected_gain = rg_vals.sum()
    total_gain_all = total_valid_gain + total_rejected_gain

    pct_retained = (
        100 * total_valid_gain / total_gain_all
        if total_gain_all > 0 else 0
    )

    summary = (
        f"Valid gain: {total_valid_gain:,.0f} km²\n"
        f"Rejected gain: {total_rejected_gain:,.0f} km²\n"
        f"Retained: {pct_retained:.1f}%"
    )

    ax.text(
        0.98,
        0.98,
        summary,
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=8,
        bbox=dict(
            facecolor="#161b22",
            edgecolor="#30363d",
            alpha=0.9,
            boxstyle="round,pad=0.4",
        ),
    )

    ax = axes[1, 1]

    gain_retention = np.where(
        total_gain > 0,
        vg_vals / total_gain * 100,
        0,
    )

    bars = ax.bar(
        x,
        gain_retention,
        color=[plt.cm.viridis(v / 100) for v in gain_retention],
        width=0.65,
    )

    ax.set_title(
        "Forest Gain Retained After Filtering (%)",
        color="#e6edf3",
    )

    ax.set_ylabel("% of Gain in Valid AOIs")
    ax.set_ylim(0, 105)

    ax.set_xticks(x)
    ax.set_xticklabels(short_labels, rotation=35, ha="right", fontsize=8)

    for bar, val in zip(bars, gain_retention):
        if val > 5:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                val + 1,
                f"{val:.0f}%",
                ha="center",
                fontsize=7,
                color="#e6edf3",
            )

    ax.yaxis.grid(True)
    ax.set_axisbelow(True)

    for ax in axes.flat:
        ax.tick_params(colors="#8b949e")

    plt.tight_layout()
    savefig(filename)


contribution_figure(
    "biome_name",
    "Biome",
    "10_biome_contribution.png",
    short_biome,
)

contribution_figure(
    "region",
    "Region",
    "11_region_contribution.png",
)

print(f"\nDone — plots saved to ./{OUT_DIR}/")

In [ ]:
import json
import logging
import math
import os
import random
import time
from pathlib import Path

import ee
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path(Path.cwd()).resolve().parent.parent / ".env")

creds_path  = os.getenv("GOOGLE_APPLICATION_CREDENTIALS", "../../ee-creds-2.json")
credentials = ee.ServiceAccountCredentials(None, creds_path)
ee.Initialize(credentials, project=os.getenv("GEE_PROJECT", "forestgrowthfoundationsbackup"))


AOI_DIR = Path("data/aois")

INPUTS = [
    (AOI_DIR / "valid_aois_enriched.json",    AOI_DIR / "valid_aois_enriched_v2.json"),
    (AOI_DIR / "rejected_aois_enriched.json", AOI_DIR / "rejected_aois_enriched_v2.json"),
]

BATCH_SIZE  = 50
MAX_RETRIES = 5


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
)
logger = logging.getLogger(__name__)

_ecoregions = ee.FeatureCollection("RESOLVE/ECOREGIONS/2017")
logger.info("GEE ready")


def atomic_write(path, obj, indent=2):
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=indent)
    tmp.replace(path)


def load_done(output_path):
    if not output_path.exists():
        return {}, set()
    with open(output_path) as f:
        existing = json.load(f)
    return {a["id"]: a for a in existing}, {a["id"] for a in existing}


def re_enrich_batch(batch):
    """
    For each AOI, intersect the cell with all overlapping ecoregions,
    compute intersection areas, and assign fields from the dominant one.
    Falls back to "Unknown" / -1 if no ecoregion overlaps at all.
    """
    features = [
        ee.Feature(
            ee.Geometry.Rectangle([a["minLon"], a["minLat"], a["maxLon"], a["maxLat"]]),
            {"id": a["id"]},
        )
        for a in batch
    ]
    fc = ee.FeatureCollection(features)

    def dominant_ecoregion(cell):
        geom = cell.geometry()

        # All ecoregions that touch this cell
        candidates = _ecoregions.filterBounds(geom)

        # For each candidate, compute the intersection area with this cell
        def with_area(eco):
            intersection = eco.geometry().intersection(geom, maxError=100)
            return eco.set("_intersect_area", intersection.area(maxError=100))

        candidates = candidates.map(with_area)

        # Pick the one with the largest intersection area
        dominant = ee.Feature(candidates.sort("_intersect_area", False).first())

        biome_name = ee.Algorithms.If(dominant, dominant.get("BIOME_NAME"), "Unknown")
        biome_num  = ee.Algorithms.If(dominant, dominant.get("BIOME_NUM"),  -1)
        region     = ee.Algorithms.If(dominant, dominant.get("REALM"),      "Unknown")

        return cell.set(
            "biome_name", biome_name,
            "biome_num",  biome_num,
            "region",     region,
        )

    updated = fc.map(dominant_ecoregion)
    return {
        f["properties"]["id"]: {
            "biome_name": f["properties"].get("biome_name", "Unknown"),
            "biome_num":  f["properties"].get("biome_num",  -1),
            "region":     f["properties"].get("region",     "Unknown"),
        }
        for f in updated.getInfo()["features"]
    }


def process_file(input_path, output_path):
    logger.info(f"\n{'─'*60}")
    logger.info(f"Input  : {input_path}")
    logger.info(f"Output : {output_path}")

    with open(input_path) as f:
        all_aois = json.load(f)
    logger.info(f"Total  : {len(all_aois):,}")

    result_map, done_ids = load_done(output_path)
    logger.info(f"Already done : {len(done_ids):,}  —  remaining : {len(all_aois) - len(done_ids):,}")

    remaining = [a for a in all_aois if a["id"] not in done_ids]
    if not remaining:
        logger.info("Nothing to do.")
        return

    # Build a lookup of original records so we can patch in new values
    original = {a["id"]: a for a in all_aois}
    total_batches = math.ceil(len(remaining) / BATCH_SIZE)
    t0 = time.time()

    for batch_num, start in enumerate(range(0, len(remaining), BATCH_SIZE), 1):
        batch = remaining[start: start + BATCH_SIZE]

        for attempt in range(MAX_RETRIES):
            try:
                updates = re_enrich_batch(batch)

                for aoi_id, new_fields in updates.items():
                    rec = dict(original[aoi_id])   # shallow copy — preserves all other fields
                    rec["biome_name"] = new_fields["biome_name"]
                    rec["biome_num"]  = new_fields["biome_num"]
                    rec["region"]     = new_fields["region"]
                    result_map[aoi_id] = rec

                atomic_write(output_path, list(result_map.values()))

                elapsed = (time.time() - t0) / 60
                logger.info(
                    f"[{batch_num}/{total_batches}] "
                    f"{len(result_map):,}/{len(all_aois):,} done "
                    f"| {elapsed:.1f} min"
                )
                time.sleep(0.2)
                break

            except Exception as e:
                err = str(e)
                retryable = any(k in err.lower() for k in
                    ["memory", "quota", "timeout", "concurrent", "429", "rate"])

                if not retryable or attempt == MAX_RETRIES - 1:
                    logger.error(f"[{batch_num}/{total_batches}] FAILED — {err}")
                    break

                wait = (2 ** attempt) + random.uniform(0, 2)
                logger.warning(
                    f"[{batch_num}/{total_batches}] retry {attempt+1}/{MAX_RETRIES} "
                    f"in {wait:.1f}s — {err[:80]}"
                )
                time.sleep(wait)

    atomic_write(output_path, list(result_map.values()), indent=2)
    logger.info(f"✓ {len(result_map):,} records → {output_path}")


if __name__ == "__main__":
    logger.info("=" * 60)
    logger.info("Area-weighted biome/region re-enrichment")
    logger.info("=" * 60)
    for inp, out in INPUTS:
        process_file(inp, out)
    logger.info("=" * 60)
    logger.info("Done")
    logger.info("=" * 60)